# Important Reminder {.unnumbered}

- This assignment must be completed using **Google Colab** and **Google Drive**. Running locally is not supported.
- You must select your **NAICS industry vertical and two rival companies before starting**. These choices carry forward to Assignments 2, 3, and 4.
- **This assignment takes significant time. Start at least one week early.**
- The corpus and vocabulary you build here will be reused in every subsequent assignment.

:::{.callout-important}
## Prerequisites

Before beginning, verify you have:

- A Google account with Google Drive accessible from Colab
- Two rival public companies in the same NAICS 4-digit vertical, each with 10-K filings on SEC EDGAR (2020–2024) and a Yahoo Finance ticker
- `yfinance`, `scikit-learn`, and `torch` available in Colab (`!pip install yfinance scikit-learn torch`)
:::



:::{.callout-important}
## DataFrame and Visualization Standard

For every DataFrame you create or transform:

- Show `.head()` to display sample rows in your notebook
- Run `.info()` to confirm column types and null counts
- Run `.describe()` to summarize numeric statistics
- Include at least one relevant visualization (bar chart, heatmap, scatter plot, or distribution plot) with a descriptive title and labeled axes

These are graded as part of your technical evidence. Undocumented DataFrames and unexplained outputs will lose points.
:::

# Submission Instructions {.unnumbered}

Submit three files on Blackboard:

1. **Word Report (`.docx`)** — maximum 10 pages, APA formatted
   - Answers to Q1–Q3 and Business Recommendation
   - Include a title page and references in APA style
   - Embed all required tables and figures inline and label them clearly
   - No code in this document

2. **Jupyter Notebook (`.ipynb`)** — complete and fully executable on Colab, organized by section
   - Include all required code, outputs, tables, and charts
   - The notebook must run top-to-bottom in a fresh Colab runtime
   - Save all required intermediate artifacts to Drive

3. **AI Disclosure Document**
   - Submit as a separate file
   - Document all AI tools used, complete prompts, output excerpts, validation steps, and share links when available
   - This is graded as part of responsible AI use and reproducibility

:::{.callout-tip}
## What Strong Submissions Consistently Do Well

- Keep the report concise and executive-facing rather than turning it into a notebook transcript
- Make every number in the report match executed notebook output exactly
- Save intermediate artifacts cleanly so later questions reuse the same corpus, chunks, vocabulary, and model outputs
- Show quality checks for extraction, section lengths, and misclassifications instead of only reporting final accuracy
- Disclose AI use specifically: which tool, for which step, with what prompt, and how the output was verified
- Preserve enough prompt-and-output context that a grader can understand your reasoning process
:::

## Word Report Expectations

Use the report to communicate business findings, not to narrate every coding step. Format the report in APA style. A strong report usually follows this structure:

1. **Executive framing (short paragraph)** — state the industry vertical, the two companies, and the business question
2. **Q1: Corpus construction** — explain what was extracted, what was filtered, and why the corpus is analytically usable
3. **Q2: Representation and model evidence** — compare TF-IDF vs. classifier results using exact metrics and figures from the notebook
4. **Q3: Reliability and failure cases** — discuss at least 3 concrete failure modes using notebook evidence
5. **Business recommendation** — make a portfolio or monitoring recommendation grounded in both text evidence and Yahoo Finance context

For full credit, your report should:

- Use exact values copied from the executed notebook, not rounded guesses or rewritten claims
- Reference the most important tables and charts directly in the surrounding prose
- Interpret the evidence in business language, not only technical language
- Present figures and tables with readable titles, labels, and captions
- Avoid contradictions between the report and the notebook
- End with a clear recommendation, limitation, and next step

## Notebook Reproducibility Standard

`Fully executable on Colab` means a grader can run the notebook from top to bottom in a fresh runtime without repairing hidden state. Your notebook should:

- Mount Drive once and define paths once near the top
- Install and import packages in a single setup section
- Execute in linear order without duplicated blocks that redefine core variables inconsistently
- Save intermediate artifacts (`company_context.csv`, `corpus.csv`, `chunks.csv`, `tfidf_matrix.csv`, `vocab.json`, model weights) before later sections reuse them
- Include short markdown cells that explain what each major section is doing and how to interpret outputs
- Show extraction-quality diagnostics such as missing-section counts, section-length summaries, and sample extracted rows
- Show model-evaluation evidence such as accuracy, confusion matrix, and a small misclassification table

Avoid notebooks that depend on manually rerunning cells out of order, local file paths, hidden variables from earlier experiments, or report claims that are not supported by executed outputs.

Suggested notebook structure:

1. Configuration, Drive mount, package installs
2. Industry and company selection — NAICS code and Yahoo Finance context table
3. SEC EDGAR data download and text extraction
4. Corpus construction: cleaning and sentence chunking
5. TF-IDF feature engineering and cosine similarity
6. PyTorch BoW classifier: training and evaluation
7. Q1–Q3 analysis and Business Recommendation

# Objective {.unnumbered}

:::{.callout-note}
This assignment introduces the end-to-end NLP pipeline for financial text analysis using real SEC filings. You may use generative AI tools for code assistance, but you must build and explain every pipeline step yourself.

**This assignment feeds Assignments 2, 3, and 4.** The corpus, vocabulary, and classifier weights you create here are reused in every subsequent assignment.
:::

:::{.callout-important}
**Core business question:** Can the vocabulary a company uses in its SEC 10-K filings reliably distinguish it from a direct competitor — and does that linguistic distinction carry financial meaning?
:::

**Business context:** An investment firm wants to determine whether two companies competing in the same NAICS vertical describe their risks differently enough to be distinguished by text alone — without reading every filing manually. Your task is to build the evidence base.

# Industry Vertical and Company Selection {#sec-setup}

## Select a NAICS Vertical

Choose one NAICS 4-digit (or 5-digit) vertical. Your vertical must have at least two publicly traded rivals with SEC 10-K filings and Yahoo Finance tickers. Find codes at the [NAICS code list](https://www.census.gov/naics/).

Justify your choice in 2–3 sentences. Example verticals that work well: regional banking (5221), cloud software (5112), specialty insurance (5241), semiconductor manufacturing (3344), pharmaceutical distribution (4242).

## Select Two Rival Companies

Your two companies must:

- Operate in the same NAICS 4-digit vertical
- Have 10-K filings for years 2020–2024 on SEC EDGAR
- Have valid Yahoo Finance tickers with market data available

## Build a Yahoo Finance Market Context Table

Pull the following fields for both tickers and store in a 2-row DataFrame:


In [ ]:
import yfinance as yf
import pandas as pd

tickers = ["TICKER_A", "TICKER_B"]
fields = ["marketCap", "trailingPE", "beta",
          "revenueGrowth", "returnOnEquity", "debtToEquity"]
rows = []
for t in tickers:
    info = yf.Ticker(t).info
    row = {f: info.get(f) for f in fields}
    row["ticker"] = t
    rows.append(row)

market_df = pd.DataFrame(rows).set_index("ticker")
print(market_df.shape)
market_df.info()
market_df.describe()
market_df


Also pull 5-year cumulative return for both tickers and add as a column.

Save to Drive:


In [ ]:
market_df.to_csv("/content/drive/MyDrive/assignment01/company_context.csv")


📊 **Required:** Show `market_df` as a table. Include it in your Word document. Briefly justify (1 paragraph) why these two companies are genuine rivals in the same vertical.

# SEC Data Acquisition and Corpus Construction {#sec-data}

## Download 10-K Filings Programmatically

Use the SEC EDGAR submissions API to retrieve the filing index for each company, then download Item 7 (MD&A), Item 7A (Quantitative and Qualitative Disclosures About Market Risk), and Item 8 (Financial Statements and Supplementary Data) for years 2020–2024.


In [ ]:
import requests
import re
import time

HEADERS = {"User-Agent": "your-email@bu.edu"}

def get_10k_index(cik: str) -> list:
    """Return list of 10-K filing metadata for a given CIK."""
    url = f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    filings = resp.json()["filings"]["recent"]
    records = []
    for i, form in enumerate(filings["form"]):
        if form == "10-K":
            records.append({
                "accession": filings["accessionNumber"][i].replace("-", ""),
                "date": filings["filingDate"][i],
                "year": int(filings["filingDate"][i][:4]),
            })
    return [r for r in records if 2020 <= r["year"] <= 2024]


For each filing, extract Item 7, Item 7A, and Item 8 text. Store each extracted section as one row in a corpus DataFrame:


In [ ]:
corpus_records = []
# For each company, each year, each item:
corpus_records.append({
    "company": company_name,
    "ticker": ticker,
    "naics_code": naics_code,
    "year": year,
    "item": item_label,       # "1A" or "7"
    "raw_text": extracted_text,
    "char_count": len(extracted_text)
})

corpus_df = pd.DataFrame(corpus_records)
print(corpus_df.shape)
corpus_df.info()
corpus_df.describe()
corpus_df.head()


📊 **Required:** Bar chart of filing count by company and year.

Save:


In [ ]:
corpus_df.to_csv("/content/drive/MyDrive/assignment01/corpus.csv", index=False)


## Build Sentence-Level Chunks

Split each `raw_text` into sentence chunks (≈ 3–5 sentences per chunk with 1-sentence overlap). Store as `chunks_df`:


In [ ]:
chunks_records = []
for _, row in corpus_df.iterrows():
    sentences = sent_tokenize(row["raw_text"])  # use nltk or spacy
    for i in range(0, len(sentences) - 2, 3):
        chunk = " ".join(sentences[i:i+4])
        chunks_records.append({
            "chunk_id": f"{row['ticker']}_{row['year']}_{row['item']}_{i}",
            "company": row["company"],
            "ticker": row["ticker"],
            "naics_code": row["naics_code"],
            "year": row["year"],
            "item": row["item"],
            "chunk_text": chunk,
            "char_count": len(chunk)
        })

chunks_df = pd.DataFrame(chunks_records)
print(chunks_df.shape)
chunks_df.info()
chunks_df.describe()
chunks_df.head()


📊 **Required:** Histogram of `char_count` by company to compare writing density.

Save:


In [ ]:
chunks_df.to_csv("/content/drive/MyDrive/assignment01/chunks.csv", index=False)


# TF-IDF Feature Engineering {#sec-tfidf}

## Build the TF-IDF Matrix


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import json

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3)
X = vectorizer.fit_transform(chunks_df["chunk_text"])
vocab = list(vectorizer.get_feature_names_out())

with open("/content/drive/MyDrive/assignment01/vocab.json", "w") as f:
    json.dump(vocab, f)

tfidf_df = pd.DataFrame(X.toarray(), columns=vocab)
tfidf_df["company"] = chunks_df["company"].values
print(tfidf_df.shape)
tfidf_df.describe().T.head(20)


## Top-20 Terms per Company

Compute the mean TF-IDF weight per term for each company. Report top-20 terms.

📊 **Required:** Side-by-side horizontal bar chart of top-20 terms for Company A and Company B. Include company names in the chart title.

Save:


In [ ]:
pd.DataFrame(X.toarray(), columns=vocab).to_csv(
    "/content/drive/MyDrive/assignment01/tfidf_matrix.csv", index=False)


## Cosine Similarity Between Matched Sections

For each year (2020–2024), compute cosine similarity between the mean TF-IDF vector for Company A's Item 7, Item 7A, and Item 8.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim_records = []
for year in range(2020, 2025):
    for item in ["1A", "7"]:
        mask_a = (chunks_df["company"] == company_a) & \
                 (chunks_df["year"] == year) & (chunks_df["item"] == item)
        mask_b = (chunks_df["company"] == company_b) & \
                 (chunks_df["year"] == year) & (chunks_df["item"] == item)
        if mask_a.sum() == 0 or mask_b.sum() == 0:
            continue
        vec_a = X[mask_a.values].mean(axis=0)
        vec_b = X[mask_b.values].mean(axis=0)
        sim = cosine_similarity(vec_a, vec_b)[0][0]
        sim_records.append({"year": year, "item": item, "cosine_similarity": sim})

sim_df = pd.DataFrame(sim_records)
print(sim_df.shape)
sim_df.info()
sim_df.describe()
sim_df.head()


📊 **Required:** Line chart of cosine similarity over years, with Item 7, Item 7A, and Item 8 as separate lines. Label axes and include a legend.

# PyTorch BoW Classifier {#sec-classifier}

## Train a BoW Linear Classifier


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

X_tensor = torch.tensor(X.toarray(), dtype=torch.float32)
y = (chunks_df["company"] == company_a).astype(int).values
y_tensor = torch.tensor(y, dtype=torch.long)

X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42, stratify=y_tensor)

class BoWClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int = 2):
        super().__init__()
        self.fc = nn.Linear(vocab_size, num_classes)

    def forward(self, x):
        return self.fc(x)

model = BoWClassifier(X_tensor.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

losses = []
for epoch in range(20):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("BoW Classifier Training Loss")
plt.show()


## Evaluate the Classifier


In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X_test).argmax(dim=1).numpy()

print(classification_report(y_test.numpy(), preds,
      target_names=[company_b, company_a]))


📊 **Required:** Confusion matrix as a seaborn heatmap with company names on axes.

Save weights:


In [ ]:
torch.save(model.state_dict(),
           "/content/drive/MyDrive/assignment01/bow_classifier_weights.pt")


## Cosine-Similarity Threshold Baseline

Predict the label of each test chunk as the label of its nearest training chunk (cosine similarity). Report baseline accuracy alongside classifier accuracy.


In [ ]:
baseline_results = pd.DataFrame({
    "Method": ["Cosine Similarity Baseline", "PyTorch BoW Classifier"],
    "Accuracy": [cosine_acc, bow_acc]})
print(baseline_results)


# Q1: Which 10-K Section Best Separates the Two Rivals? {#sec-q1}

Retrain your PyTorch BoW classifier separately on Item 7 chunks only, Item 7A chunks only, and Item 8 chunks only. Report accuracy for each run.

| Section | Accuracy |
|---------|----------|
| Item 7A – Market Risk Disclosures | |
| Item 7 – MD&A | |

📊 **Required:** Bar chart of accuracy by section.

**Answer in your Word document:** Which section produces higher classification accuracy? What does this suggest about where these two companies differentiate their language — operational risk versus strategic narrative?


# Q2: Does a Neural Classifier Outperform a Similarity Baseline? {#sec-q2}

Report the PyTorch BoW accuracy and the cosine-similarity baseline accuracy in a 2-row results table. Inspect 5 misclassified chunks (wrong-company predictions).

📊 **Required:** Show a table of 5 misclassified chunks with `chunk_text`, `true_company`, and `predicted_company` columns. Show `.head()` of this error table.

**Answer in your Word document:** What does the gap (or lack of gap) between the two methods tell you about whether company language is separated by vocabulary alone, or by something subtler?


# Q3: Do the Words That Separate the Companies Match the Numbers? {#sec-q3}

## Business Recommendation — Investment Decision

Refer to your top-20 discriminating terms bar chart (Section 4) and your Yahoo Finance market context table (Section 2).

**Answer in your Word document (≤1 page):** The top-20 discriminating terms represent each company's narrative fingerprint. Would a fundamental analyst reading only these keywords reach the same investment conclusion as the Yahoo Finance numbers do? Identify at least two terms that align with a financial metric (e.g., a term about "debt" aligning with a high debt-to-equity ratio) and at least one term that provides information the numbers do not capture. Based on this combined evidence, would you weight the two companies differently in a portfolio?


# Deliverables Summary {.unnumbered}

| Artifact | Location |
|----------|----------|
| `company_context.csv` | `/assignment01/` on Drive |
| `corpus.csv` | `/assignment01/` on Drive |
| `chunks.csv` | `/assignment01/` on Drive |
| `tfidf_matrix.csv` | `/assignment01/` on Drive |
| `vocab.json` | `/assignment01/` on Drive |
| `bow_classifier_weights.pt` | `/assignment01/` on Drive |
| Market context table | In notebook + Word doc |
| Top-20 TF-IDF terms chart | In notebook + Word doc |
| Cosine similarity over years chart | In notebook + Word doc |
| Confusion matrix heatmap | In notebook + Word doc |
| AI disclosure document | Blackboard submission |

# Use of Generative AI {.unnumbered}

You may use generative AI tools for code assistance, debugging, and explanation. You must:

- List every tool used
- State exactly which task each tool supported (for example: regex debugging, notebook refactoring, narrative editing, chart caption drafting)
- Include the prompts you submitted
- Include a shared conversation link when the tool supports sharing; if it does not, say so explicitly
- Include the complete prompt and the corresponding output excerpt or exported response, not only a one-line summary
- Identify which AI-generated outputs were used in the final submission and which were discarded or corrected
- Explain how you validated the output against notebook execution, saved artifacts, and manual inspection

When possible, preserve the original interaction using a share link or export feature. Examples include:

- **Gemini:** export the response to Google Docs or share the chat if available
- **ChatGPT:** submit a ChatGPT shared link
- **Claude:** submit a Claude shared chat link
- **Perplexity:** submit a shared thread link

If a share link is not available, include the full prompt and the relevant output in the disclosure document as screenshots or copied text.

Your AI disclosure should include the following fields:

1. Tool name and provider
2. Date used
3. Task supported
4. Complete prompt(s)
5. Share link(s), if available
6. Output excerpt(s) or exported response used in final submission
7. Validation steps you performed
8. Corrections you made after validation

Submit AI disclosure as a separate document. Generic statements such as "I used AI for help" are not sufficient for full credit.
